<style>
  .nvidia-banner {background: linear-gradient(100deg,#0b0b0b,#292929); color:white;
                  border-left:10px solid #76b900; padding:18px 22px; margin:8px 0 18px;}
  .nvidia-banner h1 {margin:0 0 6px; font-size:30px;}
  .task {border-left:6px solid #76b900; background:#f5f8f1; padding:12px 16px; margin:12px 0;}
  .checkpoint {border:1px solid #b8b8b8; border-radius:6px; padding:10px 14px; background:#fafafa;}
  .warning {border-left:6px solid #f2a900; background:#fff8e6; padding:12px 16px;}
  code {font-size: 0.92em;}
</style>

<div class="nvidia-banner">
  <h1>Module 2 — Pair with a Coding Agent: Build a ReFRAME Neighborhood Atlas</h1>
  <div>ACS Fall 2026 · Agent-assisted scientific programming · 50–65 minutes</div>
</div>

## Goal

You now know the core calls. This lesson uses one fixed scientific question:

> **How stable are the nearest structural neighbors of imatinib, linezolid, and ritonavir when the Morgan radius changes?**

Hosted mode asks Nemotron to choose the two bounded policy values for this run. Reference mode uses fixed local reference policy values with no hosted selection. In both modes, Python applies the matching allow-listed implementation. You evaluate the choices afterward, run the checks, and interpret the result.


## Setup

Use the same GPU environment as Module 1. Hosted Nemotron returns only two bounded policy choices plus explanations; Python renders, validates, and binds the function. Nemotron never returns executable Python, and no generation toggle exists.

Interactive mode requires network access and the organizer-supplied NVIDIA Inference Hub key (`sk-`) entered once in Brev Setup values. Attendees do not create a personal NVIDIA API key. For recovery, set `NVMOLKIT_WORKSHOP_MODE=reference`, restart, and rerun the notebook; reference mode makes zero client calls. Keep `workshop_llm_agent.py`, `workshop_common.py`, and `data/reframe_teaching_snapshot.csv` beside this notebook.


In [ ]:
# Prepare methods for comparing structural neighborhoods across fingerprint choices.
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from rdkit import DataStructs, RDLogger, rdBase
from rdkit.Chem import rdFingerprintGenerator

from workshop_common import add_descriptors, load_reframe

# Confirm that this notebook and its local workshop agent were released together.
import workshop_llm_agent as _workshop_llm_agent

EXPECTED_WORKSHOP_AGENT_VERSION = "2026.08.21.1"
_workshop_llm_agent = importlib.reload(_workshop_llm_agent)
loaded_agent_version = getattr(
    _workshop_llm_agent, "WORKSHOP_AGENT_VERSION", "pre-version"
)
if loaded_agent_version != EXPECTED_WORKSHOP_AGENT_VERSION:
    raise RuntimeError(
        "workshop_llm_agent.py is out of date: "
        f"expected {EXPECTED_WORKSHOP_AGENT_VERSION}, found {loaded_agent_version}. "
        "Replace the agent file with the copy distributed with this notebook, "
        "then restart the kernel and run this cell again."
    )
print(
    "Workshop agent:",
    loaded_agent_version,
    "|",
    _workshop_llm_agent.__file__,
)

RDLogger.DisableLog("rdApp.error")
SEED = 2026
np.random.seed(SEED)

# Use nvMolKit for batched fingerprint and similarity work when a GPU is available.
NVMOLKIT_READY = False
NVMOLKIT_IMPORT_ERROR = None
try:
    import torch
    import nvmolkit
    from nvmolkit.fingerprints import MorganFingerprintGenerator
    from nvmolkit.similarity import crossTanimotoSimilarity

    NVMOLKIT_READY = bool(torch.cuda.is_available())
except Exception as exc:
    NVMOLKIT_IMPORT_ERROR = repr(exc)

print(f"RDKit {rdBase.rdkitVersion}")
if NVMOLKIT_READY:
    print(f"nvMolKit {nvmolkit.__version__} | CUDA devices: {torch.cuda.device_count()}")
else:
    print("CPU teaching fallback active: nvMolKit GPU calls will be shown but evaluated with RDKit.")
    print("Reason:", NVMOLKIT_IMPORT_ERROR or "torch.cuda.is_available() is False")

In [ ]:
def fingerprint_tensor(result):
    """Return the CUDA torch tensor wrapped by an nvMolKit fingerprint result."""
    return result if isinstance(result, torch.Tensor) else result.torch()


def gpu_to_numpy(result):
    """Synchronize an nvMolKit result or CUDA tensor and return a host array."""
    if isinstance(result, torch.Tensor):
        return result.detach().cpu().numpy()
    return result.numpy()


# Morgan fingerprints describe local atomic environments at a chosen radius.
def make_fingerprints(molecules, radius=2, fp_bits=1024):
    """Return nvMolKit CUDA fingerprints, or RDKit fingerprints in fallback mode."""
    if NVMOLKIT_READY:
        generator = MorganFingerprintGenerator(radius=radius, fpSize=fp_bits)
        return fingerprint_tensor(generator.GetFingerprints(list(molecules), num_threads=0))
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_bits)
    return generator.GetFingerprints(list(molecules), numThreads=0)


# Tanimoto similarity compares the structural features encoded by two fingerprints.
def tanimoto_matrix(first, second=None):
    """Return a host similarity matrix from the active backend."""
    if NVMOLKIT_READY:
        return gpu_to_numpy(crossTanimotoSimilarity(first, second))
    second = first if second is None else second
    return np.asarray(
        [DataStructs.BulkTanimotoSimilarity(query, second) for query in first],
        dtype=float,
    )


print("Shared ReFRAME helpers and Module 2 fingerprint helpers are ready.")

### What the notebook computes

- **Morgan fingerprints** encode circular atom environments as a fixed-length bit vector. Radius changes how far each local environment extends; fingerprint length changes the collision budget.
- **Tanimoto similarity** compares two binary fingerprints. It is a structural-neighborhood measure, not a measurement of target binding or biological activity.
- **Radius comparison** uses top-neighbor overlap to show representation sensitivity.

The notebook uses nvMolKit when a compatible NVIDIA GPU is available. Its CPU fallback exists only so the teaching narrative and checks remain inspectable on a non-GPU laptop; the workshop's accelerated exercises should report `NVMOLKIT_READY == True`.


In [ ]:
# Hold the snapshot sample and reference compounds fixed so radius is the main comparison.
SAMPLE_SIZE = 96
ANCHOR_TERMS = ("imatinib", "linezolid", "ritonavir")
RADII = (2, 3)
FP_BITS = 1024
TOP_K = 10

reframe = add_descriptors(load_reframe(SAMPLE_SIZE, anchor_terms=ANCHOR_TERMS, source="snapshot"))
print("Source:", reframe.attrs["source"])
print("Rows:", len(reframe), "| Backend:", "nvMolKit GPU" if NVMOLKIT_READY else "RDKit CPU fallback")


## Step 1 — Give Nemotron a bounded policy request

In hosted mode, the next cell asks Nemotron only to choose two failure policies and explain them. Reference mode uses fixed local policy values without a client call. Python owns the implementation and tests.


In [ ]:
# Ask Nemotron only for the bounded policy choices.
AGENT_PROMPT = f"""
Select two bounded policies for a fixed ACS teaching run.
The run compares top {TOP_K} non-self structural neighbors for {ANCHOR_TERMS}
in a fixed ReFRAME snapshot using Morgan radii {RADII} and {FP_BITS} bits.

`MISSING_ANCHOR`: `raise` stops when an anchor is absent; `skip` continues with found anchors.
`INVALID_MATRIX`: `raise` stops on an invalid similarity matrix; `skip` omits that radius.
Choose one value for each policy and give two concise explanations.
This is a structural-neighborhood exercise, not evidence of biological activity.
Return only the required policy schema and no code.
""".strip()

display(Markdown("```text\n" + AGENT_PROMPT + "\n```"))


## Step 2 — Inspect the policy and locally rendered function

The notebook prints its exact mode once. Reference uses a fixed valid policy without a client call. Interactive mode sends only the bounded policy prompt to Nemotron; it does not send dataset rows, rendered code, or analysis results. Python renders the executable function locally.

The displayed source is rendered locally from validated policy values and then executed by the same bounded helper used for the acceptance checks.

The policy receipt is plain text. The displayed function is Python-owned renderer output, not hosted code.


In [ ]:
# Hosted mode uses Nemotron-selected values; reference mode uses fixed local values.
# Select exactly one mode. Reference is deterministic; interactive needs a protected key.
WORKSHOP_ROOT = Path(_workshop_llm_agent.__file__).resolve().parent
WORKSHOP_MODE = _workshop_llm_agent.workshop_mode()
print(f"NVMOLKIT_WORKSHOP_MODE={WORKSHOP_MODE}")
implementation = _workshop_llm_agent.select_neighborhood_implementation(
    AGENT_PROMPT, mode=WORKSHOP_MODE
)
print("Implementation label:", implementation.label)
display(Markdown("```python\n" + implementation.function_source + "\n```"))
print("Missing anchor explanation:", implementation.policy.missing_anchor_explanation)
print("Invalid matrix explanation:", implementation.policy.invalid_matrix_explanation)


## Step 3 — Validate and bind the locally rendered function

The helper validates that the source exactly matches the local renderer output for the selected policy, then binds `build_neighborhood_atlas`. There is no attendee copy or paste step.

The following cell runs the selected locally rendered function on the fixed ReFRAME snapshot and checks its normal-path invariants.


In [ ]:
# The helper executes only the exact Python source rendered from validated policy values.
atlas_builder = _workshop_llm_agent.bind_neighborhood_builder(
    implementation,
    {"np": np, "pd": pd, "make_fingerprints": make_fingerprints, "tanimoto_matrix": tanimoto_matrix},
)
print("Implementation under test:", implementation.label)


## Step 4 — Run the bound function and its normal-path invariant checks

This cell checks the valid fixed run. It does not trigger the selected missing-anchor or invalid-matrix failure branches.


In [ ]:
# Check the bounded implementation on the fixed valid run.
print("Implementation under test:", implementation.label)

atlas = atlas_builder(
    reframe, ANCHOR_TERMS, radii=RADII, fp_bits=FP_BITS, top_k=TOP_K
)

expected_columns = [
    "radius", "query", "query_ikey", "rank", "neighbor",
    "neighbor_ikey", "tanimoto", "profile"
]
assert list(atlas.columns) == expected_columns
assert len(atlas) == len(RADII) * len(ANCHOR_TERMS) * TOP_K
assert atlas["tanimoto"].between(0, 1).all()
assert (atlas["query_ikey"] != atlas["neighbor_ikey"]).all()
assert atlas.groupby(["radius", "query"])["neighbor_ikey"].nunique().eq(TOP_K).all()
assert atlas.groupby(["radius", "query"])["rank"].apply(list).apply(lambda x: x == list(range(1, TOP_K + 1))).all()
print("✓ Normal-path invariant checks passed for the fixed valid run.")
print("Selected failure branches were not triggered by this fixed valid run.")
attendee_columns = ["radius", "query", "rank", "neighbor", "tanimoto"]
attendee_atlas = atlas.loc[:, attendee_columns].round({"tanimoto": 3})
display(attendee_atlas.head(12))


## Step 5 — Compare the methods, not just the molecules

We use Jaccard overlap of the two top-10 sets. `1.0` means identical membership; `0.0` means no shared neighbors. Rank order can still change even when membership is identical.


In [ ]:
# Jaccard overlap shows how much each neighbor set changes with fingerprint radius.
overlap_rows = []
radius_a, radius_b = RADII
for query in atlas["query"].unique():
    set_a = set(atlas.query("radius == @radius_a and query == @query")["neighbor_ikey"])
    set_b = set(atlas.query("radius == @radius_b and query == @query")["neighbor_ikey"])
    overlap_rows.append({
        "query": query,
        f"radius_{radius_a}_only": len(set_a - set_b),
        "shared": len(set_a & set_b),
        f"radius_{radius_b}_only": len(set_b - set_a),
        "jaccard": len(set_a & set_b) / len(set_a | set_b),
    })

overlap = pd.DataFrame(overlap_rows).sort_values("jaccard")
display(overlap.round(3))
overlap.plot.bar(x="query", y="jaccard", ylim=(0, 1), color="#76b900", legend=False, figsize=(8, 3.5))
plt.ylabel("Top-10 Jaccard overlap")
plt.title("Neighborhood stability when Morgan radius changes")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## Step 6 — Review the local receipt and results

The receipt shows the two bounded policy values used for this run. The lesson does not send rendered code or analysis results to a hosted client.

Review the receipt and the local normal-path invariant results, then add one useful local test.


In [ ]:
# The two policy explanations are the complete bounded policy receipt.
print("Missing anchor policy:", implementation.policy.missing_anchor)
print("Invalid matrix policy:", implementation.policy.invalid_matrix)
if implementation.label == "hosted_nemotron":
    print("Hosted mode: Nemotron chose the two bounded policy values for this run.")
elif implementation.label == "reference":
    print("Reference mode: fixed local reference policy values were used; no hosted selection occurred.")
else:
    raise AssertionError(f"Unexpected implementation label: {implementation.label}")
print("Python applies the matching allow-listed implementation.")
print("You evaluate the choices afterward, run the checks, and interpret the result.")
print("This lesson does not send code or results back to a hosted client.")


In [ ]:
# Participant test: confirm each neighborhood runs from most to least similar.
monotonic = atlas.groupby(["radius", "query"])["tanimoto"].apply(
    lambda values: values.is_monotonic_decreasing
)
assert monotonic.all()
print("✓ Review test passed: every neighborhood is sorted by descending similarity.")


## Checks and next step

Discuss selected failure policies, representation sensitivity, and scientific interpretation with a partner:

1. Are both selected policies appropriate? If not, which values would you choose and why?
2. Which radius-sensitive anchor has the lowest Jaccard overlap, and what structural feature might explain that?
3. Which unsupported scientific inference must these neighborhoods not support?

In Module 3, you will use a bounded panel-design agent with an objective, boundaries, and acceptance tests.


## Sources and scientific boundary

- [nvMolKit repository](https://github.com/NVIDIA-BioNeMo/nvMolKit)
- [nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/)
- [Installed workshop nvMolKit skill](../skills/nvmolkit/SKILL.md) — authoritative for this environment
- [reframeDb](https://reframedb.org/) and its public `reframe_smiles_list.csv` export

The ReFRAME export is used for teaching and should be handled under the site's current terms. Refresh it before delivery and do not treat availability status as evidence of clinical suitability. The fingerprints, Tanimoto neighborhoods, and radius comparison run here do **not** establish binding, activity, ADMET, efficacy, safety, synthesizability, or experimental structure.


In [ ]:
# The lowest overlap marks the neighborhood most sensitive to representation choice.
answer_sensitivity = overlap.sort_values(["jaccard", "query"], ascending=[True, True]).reset_index(drop=True)
print("Most radius-sensitive anchor = lowest top-10 Jaccard overlap")
display(answer_sensitivity.round(3))

most_sensitive_query = answer_sensitivity.iloc[0]["query"]
most_sensitive_jaccard = float(answer_sensitivity.iloc[0]["jaccard"])
print(f"Answer for this run: {most_sensitive_query} (Jaccard = {most_sensitive_jaccard:.3f})")


## Answer key — discussion checkpoint

<details>
<summary><b>Reveal instructor key</b></summary>

1. **Selected failure policies:** `raise`/`raise` is appropriate for the fixed teaching run, while the recorded run values remain the values actually used by that run. A missing anchor or invalid matrix would make results partial or misaligned. A selected `skip` would continue with found anchors or omit the affected radius, changing which results are reported.

2. **Most radius-sensitive anchor:** The correct answer for the current run is the first row of `answer_sensitivity`, which has the lowest top-10 Jaccard overlap. A plausible explanation must reference actual structural differences among the moved neighbors. Radius 3 incorporates larger connected environments than radius 2; therefore scaffold context and substitution patterns can displace molecules that only share smaller local motifs. Do not award full credit for merely saying “radius 3 is more accurate”—it is a different representation, not universally more correct.

3. **Unsupported scientific inference:** These structural neighborhoods do not establish binding, activity, ADMET, efficacy, or safety. They only support a representation-sensitivity discussion.

**Review-test key:** The supplied monotonicity test is useful because each neighborhood must be ordered by decreasing similarity. Additional strong tests include deterministic tie handling, missing-anchor behavior, duplicate connectivity keys, and agreement with a small independently computed RDKit example.

</details>
